# 02 · 双塔召回模型：训练与评估

前置：已运行 `01_data_prep.ipynb`，`OUT_DIR`（默认 `MyDrive/two_tower_movies_tv/`）里有
`train/valid/test_data.npz` 和 `id_maps.npz`。

- **用户塔**：用户 ID embedding + 最近 20 个历史商品 embedding 平均（无穿越）
- **物品塔**：商品 ID embedding + 标题词 embedding 平均 + 类目 embedding 平均
- **损失**：in-batch softmax + logQ 流行度纠偏 + 批内重复商品屏蔽
- **评估**：Recall@10 / Recall@50 / NDCG@10，对比热门基线

> 先在菜单「代码执行程序 → 更改运行时类型」选 **T4 GPU**，再运行。

In [ ]:
# ---------------- 0. 环境 & 配置 ----------------
import os, gzip, json, re, time, shutil, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import tensorflow as tf
import keras

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "无（使用 CPU）")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive"                       # 原始 meta 文件所在位置
    OUT_DIR   = "/content/drive/MyDrive/two_tower_movies_tv"   # leave-last-out 的输出
    LOCAL_DIR = "/content/amazon2023"
else:
    DRIVE_DIR = LOCAL_DIR = os.environ.get("DATA_DIR", "./data/amazon2023")
    OUT_DIR = os.environ.get("OUT_DIR", DRIVE_DIR)

CFG = dict(
    category      = "Movies_and_TV",
    max_hist      = 20,       # 必须和 leave_last_out 里一致
    max_title_len = 16,       # 标题最多保留的词数
    max_cats      = 8,        # 每个商品最多保留的类目数
    title_vocab   = 50_000,   # 标题词表大小
    emb_dim       = 64,
    temperature   = 0.05,
    batch_size    = 4096,
    epochs        = 3,
    lr            = 1e-3,
    eval_users    = 50_000,   # 评估时抽样的用户数（None = 全量，较慢）
    seed          = 42,
)
os.makedirs(LOCAL_DIR, exist_ok=True)
keras.utils.set_random_seed(CFG["seed"])

In [ ]:
# ---------------- 1. 读取 leave-last-out 数据 ----------------
t0 = time.time()
d = {s: dict(np.load(f"{OUT_DIR}/{s}_data.npz")) for s in ("train", "valid", "test")}
train_data, valid_data, test_data = d["train"], d["valid"], d["test"]
maps = np.load(f"{OUT_DIR}/id_maps.npz", allow_pickle=True)
user_ids, item_ids, item_freq = maps["user_ids"], maps["item_ids"], maps["item_freq"]
N_USERS, N_ITEMS = len(user_ids), len(item_ids)

assert train_data["hist"].shape[1] == CFG["max_hist"], "max_hist 和数据准备时不一致"
for name, dd in [("train", train_data), ("valid", valid_data), ("test", test_data)]:
    print(f"{name:5s}: {len(dd['user']):,}")
print(f"用户 {N_USERS:,} | 商品 {N_ITEMS:,} | 读取用时 {time.time()-t0:.0f}s")

# logQ 纠偏用的商品概率（item_freq 只用 train 统计）
item_logq = np.log((item_freq + 1) / (item_freq.sum() + N_ITEMS + 1)).astype("float32")

# 元数据复制到 Colab 本地磁盘再读
meta_name = f"meta_{CFG['category']}.jsonl.gz"
meta_path = os.path.join(LOCAL_DIR, meta_name)
if not os.path.exists(meta_path):
    if os.path.exists(os.path.join(DRIVE_DIR, meta_name)):
        shutil.copy(os.path.join(DRIVE_DIR, meta_name), meta_path)
    else:
        url = f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/{meta_name}"
        print("下载", url)
        urllib.request.urlretrieve(url, meta_path)

In [ ]:
# ---------------- 2. 商品元数据（标题 + 类目） ----------------
item_index = {asin: idx + 1 for idx, asin in enumerate(item_ids)}
titles = [""] * (N_ITEMS + 1)
cats   = [[] for _ in range(N_ITEMS + 1)]
GENERIC_CATS = {"Movies & TV", "Prime Video", "Featured Categories",
                "Genre for Featured Categories", "Movies", "TV"}

t0, matched = time.time(), 0
with gzip.open(meta_path, "rt", encoding="utf-8") as f:
    for line in f:
        m = json.loads(line)
        idx = item_index.get(m.get("parent_asin"))
        if idx is None:
            continue
        matched += 1
        titles[idx] = (m.get("title") or "").strip()
        c = [x for x in (m.get("categories") or []) if x and x not in GENERIC_CATS]
        cats[idx] = c[: CFG["max_cats"]]
print(f"元数据匹配 {matched:,}/{N_ITEMS:,} 个商品，用时 {time.time()-t0:.0f}s")
print(f"有标题的商品: {sum(bool(t) for t in titles):,}")

# 标题分词 -> 整数矩阵 [N_ITEMS+1, max_title_len]，0 = padding
tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
title_tokens = [tok(t) for t in titles]
word_cnt = Counter(w for ws in title_tokens for w in ws)
word_vocab = {w: j + 1 for j, (w, _) in enumerate(word_cnt.most_common(CFG["title_vocab"]))}
L = CFG["max_title_len"]
title_mat = np.zeros((N_ITEMS + 1, L), dtype=np.int32)
for idx, ws in enumerate(title_tokens):
    ids = [word_vocab[w] for w in ws if w in word_vocab][:L]
    title_mat[idx, :len(ids)] = ids

# 类目 -> 整数矩阵 [N_ITEMS+1, max_cats]
cat_cnt = Counter(c for cs in cats for c in cs)
cat_vocab = {c: j + 1 for j, (c, n_) in enumerate(cat_cnt.most_common()) if n_ >= 5}
C = CFG["max_cats"]
cat_mat = np.zeros((N_ITEMS + 1, C), dtype=np.int32)
for idx, cs in enumerate(cats):
    ids = [cat_vocab[c] for c in cs if c in cat_vocab][:C]
    cat_mat[idx, :len(ids)] = ids

print(f"标题词表 {len(word_vocab):,} | 类目词表 {len(cat_vocab):,}")
print("常见类目:", [c for c, _ in cat_cnt.most_common(10)])

In [ ]:
# ---------------- 3. 双塔模型 ----------------
def masked_mean(emb, ids):
    # emb: [B, T, D], ids: [B, T]（0 为 padding）
    mask = tf.cast(ids > 0, emb.dtype)[..., None]
    return tf.reduce_sum(emb * mask, axis=1) / tf.maximum(tf.reduce_sum(mask, axis=1), 1.0)


class ItemTower(keras.Model):
    def __init__(self, n_items, title_mat, cat_mat, n_words, n_cats, dim):
        super().__init__()
        self.title_mat = tf.constant(title_mat)
        self.cat_mat = tf.constant(cat_mat)
        self.item_emb = keras.layers.Embedding(n_items + 1, dim, name="item_id_emb")
        self.word_emb = keras.layers.Embedding(n_words + 1, dim, name="title_word_emb")
        self.cat_emb = keras.layers.Embedding(n_cats + 1, dim, name="category_emb")
        self.mlp = keras.Sequential([keras.layers.Dense(dim * 2, activation="gelu"),
                                     keras.layers.Dense(dim)])

    def call(self, item_idx):
        t = tf.gather(self.title_mat, item_idx)
        c = tf.gather(self.cat_mat, item_idx)
        x = tf.concat([self.item_emb(item_idx),
                       masked_mean(self.word_emb(t), t),
                       masked_mean(self.cat_emb(c), c)], axis=-1)
        return tf.math.l2_normalize(self.mlp(x), axis=-1)


class UserTower(keras.Model):
    def __init__(self, n_users, item_id_emb, dim):
        super().__init__()
        self.user_emb = keras.layers.Embedding(n_users + 1, dim, name="user_id_emb")
        self.item_id_emb = item_id_emb      # 与物品塔共享商品 ID embedding
        self.mlp = keras.Sequential([keras.layers.Dense(dim * 2, activation="gelu"),
                                     keras.layers.Dense(dim)])

    def call(self, user_idx, hist):
        x = tf.concat([self.user_emb(user_idx),
                       masked_mean(self.item_id_emb(hist), hist)], axis=-1)
        return tf.math.l2_normalize(self.mlp(x), axis=-1)


item_tower = ItemTower(N_ITEMS, title_mat, cat_mat, len(word_vocab), len(cat_vocab), CFG["emb_dim"])
user_tower = UserTower(N_USERS, item_tower.item_emb, CFG["emb_dim"])
_ = item_tower(tf.constant([1])); _ = user_tower(tf.constant([1]), tf.zeros((1, CFG["max_hist"]), tf.int32))
n_params = sum(int(np.prod(v.shape)) for v in
               {id(v): v for v in item_tower.trainable_variables + user_tower.trainable_variables}.values())
print(f"参数量（共享 embedding 只计一次）: ~{n_params/1e6:.1f}M")

In [ ]:
# ---------------- 4. 训练（in-batch softmax + logQ 纠偏） ----------------
logq_table = tf.constant(item_logq)
optimizer = keras.optimizers.Adam(CFG["lr"])
variables = list({id(v): v for v in user_tower.trainable_variables + item_tower.trainable_variables}.values())

def make_ds(dd, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((dd["user"], dd["hist"], dd["item"]))
    if shuffle:
        ds = ds.shuffle(min(len(dd["user"]), 1_000_000), seed=CFG["seed"], reshuffle_each_iteration=True)
    return ds.batch(CFG["batch_size"], drop_remainder=shuffle).prefetch(tf.data.AUTOTUNE)

@tf.function
def train_step(u, h, i):
    with tf.GradientTape() as tape:
        ue = user_tower(u, h)                         # [B, D]
        ie = item_tower(i)                            # [B, D]
        logits = tf.matmul(ue, ie, transpose_b=True) / CFG["temperature"]
        logits -= tf.gather(logq_table, i)[None, :]   # logQ 流行度纠偏
        # 批内同一商品出现多次时，非对角位置不当作负样本
        same = tf.equal(i[:, None], i[None, :])
        diag = tf.eye(tf.shape(i)[0], dtype=tf.bool)
        logits = tf.where(same & ~diag, -1e9, logits)
        labels = tf.range(tf.shape(i)[0])
        loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels, logits))
    grads = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(grads, variables))
    return loss


def all_item_embeddings(bs=8192):
    idx = np.arange(1, N_ITEMS + 1, dtype=np.int32)
    embs = [item_tower(tf.constant(idx[s:s + bs])) for s in range(0, len(idx), bs)]
    return tf.concat(embs, axis=0)                    # 第 r 行对应商品 r+1


def evaluate(dd, ks=(10, 50), max_users=CFG["eval_users"], bs=1024):
    rng = np.random.default_rng(CFG["seed"])
    sel = np.arange(len(dd["user"]))
    if max_users and len(sel) > max_users:
        sel = rng.choice(sel, max_users, replace=False)
    item_E = all_item_embeddings()
    kmax = max(ks)
    hits = {k: 0.0 for k in ks}; ndcg10 = 0.0
    for s in range(0, len(sel), bs):
        b = sel[s:s + bs]
        ue = user_tower(tf.constant(dd["user"][b]), tf.constant(dd["hist"][b]))
        scores = tf.matmul(ue, item_E, transpose_b=True)
        top = tf.math.top_k(scores, k=kmax).indices.numpy() + 1   # 转回商品编号
        match = top == dd["item"][b][:, None]
        for k in ks:
            hits[k] += match[:, :k].any(axis=1).sum()
        pos = np.argmax(match[:, :10], axis=1)
        found = match[:, :10].any(axis=1)
        ndcg10 += (found / np.log2(pos + 2)).sum()
    m = len(sel)
    res = {f"Recall@{k}": hits[k] / m for k in ks}
    res["NDCG@10"] = ndcg10 / m
    return res


train_ds = make_ds(train_data, shuffle=True)
steps = len(train_data["user"]) // CFG["batch_size"]
history, best_score, best_epoch, best_weights = [], -1.0, 0, None
for epoch in range(1, CFG["epochs"] + 1):
    t0, total = time.time(), 0.0
    for step, (u, h, i) in enumerate(train_ds, 1):
        total += float(train_step(u, h, i))
        if step % 200 == 0 or step == steps:
            print(f"epoch {epoch} step {step}/{steps}  loss {total/step:.4f}  {time.time()-t0:.0f}s", end="\r")
    print()
    val = {k: round(float(v), 4) for k, v in evaluate(valid_data).items()}
    history.append({"epoch": epoch, "loss": round(total / step, 4), **val})
    print(f"epoch {epoch} 验证集:", val)
    # 只保留验证集 Recall@50 最好的那一轮权重（防止后面的 epoch 过拟合拖低结果）
    if val["Recall@50"] > best_score:
        best_score, best_epoch = val["Recall@50"], epoch
        best_weights = [v.numpy() for v in variables]

for v, w in zip(variables, best_weights):
    v.assign(w)
print(f"使用第 {best_epoch} 个 epoch 的权重（验证集 Recall@50 = {best_score:.4f}）")

In [ ]:
# ---------------- 5. 测试集评估 + 热门基线 ----------------
test_metrics = evaluate(test_data)
print("测试集:", {k: round(float(v), 4) for k, v in test_metrics.items()})

pop_top = np.argsort(-item_freq)[:50]
pop_hit10 = np.isin(test_data["item"], pop_top[:10]).mean()
pop_hit50 = np.isin(test_data["item"], pop_top[:50]).mean()
print(f"热门基线: Recall@10 {pop_hit10:.4f} | Recall@50 {pop_hit50:.4f}")

# 结果写进 metrics.json，方便填 README
metrics = {"config": CFG, "best_epoch": best_epoch, "history": history,
           "test": {k: round(float(v), 4) for k, v in test_metrics.items()},
           "popularity_baseline": {"Recall@10": round(float(pop_hit10), 4),
                                   "Recall@50": round(float(pop_hit50), 4)}}
with open(f"{OUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(json.dumps(metrics["test"], ensure_ascii=False))

In [ ]:
# ---------------- 6. 看看推荐结果 ----------------
item_E = all_item_embeddings()

def show_items(idxs, header):
    print(header)
    for r, idx in enumerate(idxs, 1):
        t = titles[idx] or "(无标题)"
        print(f"  {r:2d}. {t[:80]}  [{item_ids[idx-1]}]")

def recommend_for_user(row=0, k=10, dd=test_data):
    u, h, target = dd["user"][row], dd["hist"][row], dd["item"][row]
    show_items([x for x in h if x > 0][:5], f"用户 {user_ids[u-1]} 最近看过：")
    ue = user_tower(tf.constant([u]), tf.constant([h]))
    scores = tf.matmul(ue, item_E, transpose_b=True)[0].numpy()
    scores[[x - 1 for x in h if x > 0]] = -np.inf          # 不推荐看过的
    top = np.argsort(-scores)[:k] + 1
    show_items(top, f"\nTop-{k} 推荐：")
    print(f"\n真实下一个：{titles[target] or '(无标题)'}  {'✅ 命中' if target in top else ''}")

def similar_items(query, k=10):
    q = query.lower()
    cand = [j for j in range(1, N_ITEMS + 1) if q in titles[j].lower()]
    if not cand:
        print("没找到包含该关键词的商品"); return
    j = max(cand, key=lambda x: item_freq[x])              # 取最热门的匹配项
    scores = tf.linalg.matvec(item_E, item_E[j - 1]).numpy()
    top = [x + 1 for x in np.argsort(-scores) if x + 1 != j][:k]
    show_items(top, f"与《{titles[j]}》相似的商品：")

recommend_for_user(row=0)
print("\n" + "-" * 60)
similar_items("star wars")

In [ ]:
# ---------------- 7. 保存 embedding 和模型权重到 Drive ----------------
np.save(f"{OUT_DIR}/item_embeddings.npy", item_E.numpy())
pd.DataFrame({"item_idx": np.arange(1, N_ITEMS + 1), "parent_asin": item_ids,
              "title": titles[1:]}).to_csv(f"{OUT_DIR}/items.csv", index=False)
item_tower.save_weights(f"{OUT_DIR}/item_tower.weights.h5")
user_tower.save_weights(f"{OUT_DIR}/user_tower.weights.h5")
print("已保存到", OUT_DIR)